In [ ]:

import pandas as pd
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

Base = "https://www.aladin.co.kr/"


def extract_page(li_list):
    for li in li_list:
        li_text = li.get_text(" ", strip=True)
        page = ""
        if '쪽' in li_text:
            idx = li_text.find('쪽')
            i = idx-1
            while i>=0 and li_text[i].isdigit():
                page = li_text[i] + page
                i -= 1
        if page:
            return int(page)       
    return None


def distinct_page(soup):
    return soup.find("div", attrs={"class":"Ere_prod_mconts_R"}) or soup.find("div", attrs={"class":"Ere_prod_mconts_box"})


#페이지 찾기 함수: isbn 입력 -> 페이지 추출
def find_bookpage(isbn):
    #isbn을 통해 검색 페이지로 이동
    headers = {"User-Agent":"Mozilla/5.0"}
    url = r"https://www.aladin.co.kr/search/wsearchresult.aspx?SearchTarget=Book&SearchWord={}"
    try:
        page_1 = requests.get(url.format(isbn), headers=headers, timeout=10)
        page_1.raise_for_status()
    except (requests.HTTPError, requests.RequestException) as err:
        return None, (f"[FAIL]...isbn={isbn}, error={err}")
    
    #뷰티풀수프 객체 만들기, 상세페이지 이동 경로 설정
    page_1_soup = BeautifulSoup(page_1.text, 'html.parser')
    link_1 = page_1_soup.find("a", attrs={'class':'bo3'})
    if link_1 is None:
        return None, f"No_bo3_link{isbn}"
    link_2 = link_1.get('href')
    if not link_2:
        return None, f"no_href{isbn}"
    
    #상세페이지 링크에서 쪽수 객체 찾기
    try: 
        page_2 = requests.get(link_2, headers=headers, timeout=10)
        page_2.raise_for_status()
    except (requests.HTTPError, requests.RequestException) as err:
        return None, f"[FAIL]...isbn={isbn}, error={err}"
    
    page_2_soup = BeautifulSoup(page_2.text, 'html.parser')
    page_box = page_2_soup.find("div", attrs={"class":"Ere_prod_mconts_R"}) or page_2_soup.find("div", attrs={"class":"Ere_prod_mconts_box"})
    if page_box is None:
        return None, f"no_page_box{isbn}"
    
    li_list = [li for ul in page_box.find_all('ul') for li in ul.find_all('li')]
    if not li_list:
        return None, f"no_li_list{isbn}"
    
    for li in li_list:
        li_text = li.get_text(" ", strip=True)
        page = ""
        if '쪽' in li_text:
            idx = li_text.find('쪽')
            i = idx-1
            while i>=0 and li_text[i].isdigit():
                page = li_text[i] + page
                i -= 1
        if page:
            return int(page), None
        
    return None, f"no_page{isbn}"
            

df = pd.read_json(r"C:\data\2030s_bestbook_df.json", encoding='utf-8')
tmp = df['isbn13'].apply(find_bookpage)
tmp_list = tmp.tolist()
df[['page_count', 'fail_reason']] = pd.DataFrame(tmp_list, index = df.index)

In [51]:
sum(df['fail_reason'].notna())

1

In [ ]:
import pandas as pd

df2 = pd.DataFrame({
    "book": ["A", "B", "C", "D", "E", "F"],
    "page_count": [320, None, 250, np.nan, 180, 400],
    "rating": [4.5, 4.0, None, 3.5, np.nan, 4.8],
})

for i in df['fail_reason']:
    if i is None:
        pass
    else:
        print(i)
        


SyntaxError: closing parenthesis ')' does not match opening parenthesis '[' (529998544.py, line 15)